# Análise Exploratória de Dados (EDA) - Telco Customer Churn

Este notebook investiga a base **Telco Customer Churn** antes da etapa de modelagem. A ideia é entender a estrutura dos dados, corrigir inconsistências simples e levantar padrões de churn que depois podem orientar tanto a rede neural quanto o dashboard no Power BI.

A base original tem **7.043 clientes** e **21 variáveis**. A variável-alvo é `Churn`, que indica se o cliente cancelou (`Yes`) ou permaneceu (`No`).


## 1. Preparação do ambiente

Neste primeiro bloco, vamos importar o `pandas`, biblioteca principal para carregar, transformar e resumir a base tabular. Ela será usada em praticamente todas as etapas da EDA.


In [9]:
import pandas as pd

## 2. Carregamento e inspeção inicial

Aqui vamos ler o CSV original da pasta `data`, verificar a estrutura com `df.info()` e visualizar as primeiras linhas com `df.head()`. Esse passo confirma dimensões, tipos de dados, presença de nulos explícitos e exemplos reais de valores em cada coluna.


In [10]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


A base foi carregada com **7.043 linhas** e **21 colunas**, exatamente como esperado para o dataset Telco. O `info()` mostra que não há valores nulos explícitos, mas também revela um ponto importante: `TotalCharges` aparece como `str`, embora represente um valor financeiro acumulado. Isso indica que existe sujeira nessa coluna e que ela precisa ser convertida antes de entrar em correlações, gráficos numéricos ou modelos.

Também vale notar que `customerID` é apenas identificador do cliente. Ele é útil para rastrear clientes e montar arquivos para o Power BI, mas não deve ser tratado como variável preditiva no modelo.


## 3. Tratamento de `TotalCharges`

Neste bloco, vamos converter `TotalCharges` para formato numérico. Qualquer valor que não puder ser convertido vira `NaN` com `errors="coerce"`, e em seguida esses casos são preenchidos com `0`. Esses registros correspondem aos clientes com pouco ou nenhum tempo de permanência, em que a cobrança total ainda não foi acumulada.


In [11]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

A célula não imprime saída, mas altera o dataframe em memória: `TotalCharges` deixa de ser texto e passa a poder ser usado como número. Esse ajuste é essencial porque evita erros em cálculos estatísticos e no pré-processamento do modelo. Também mantém a interpretação de negócio dos registros em branco: clientes recém-entrados podem ter cobrança total igual a zero.


## 4. Distribuição da variável-alvo

Agora vamos medir a proporção de clientes que cancelaram (`Yes`) e que permaneceram (`No`). Essa checagem mostra se o problema de classificação está balanceado ou se uma classe aparece com muito mais frequência que a outra.


In [12]:
df["Churn"].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

O resultado mostra aproximadamente **73,46% de clientes sem churn** e **26,54% com churn**. Portanto, a base é desbalanceada: a maioria dos clientes permanece, enquanto a classe de interesse comercial (`Yes`) é menor.

Isso afeta a modelagem. Uma acurácia alta pode ser enganosa se o modelo simplesmente favorecer a classe majoritária. Para a rede neural, faz sentido acompanhar métricas como **AUC**, **recall da classe churn** e possivelmente usar `class_weight`.


## 5. Churn por segmentos de negócio

Neste bloco, vamos comparar a taxa de churn em três recortes importantes para o negócio: tipo de contrato, faixa de tempo de permanência e tipo de serviço de internet. Também criamos `tenure_group`, uma versão categórica de `tenure`, para facilitar leituras por faixa e uso posterior no Power BI.


In [13]:
print(df.groupby("Contract")["Churn"].value_counts(normalize=True))
df["tenure_group"] = pd.cut(df["tenure"], bins=[0, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49+"], include_lowest=True)
print(df.groupby("tenure_group")["Churn"].value_counts(normalize=True))
print(df.groupby("InternetService")["Churn"].value_counts(normalize=True))


Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887305
                Yes      0.112695
Two year        No       0.971681
                Yes      0.028319
Name: proportion, dtype: float64
tenure_group  Churn
0-12          No       0.525618
              Yes      0.474382
13-24         No       0.712891
              Yes      0.287109
25-48         No       0.796110
              Yes      0.203890
49+           No       0.904868
              Yes      0.095132
Name: proportion, dtype: float64
InternetService  Churn
DSL              No       0.810409
                 Yes      0.189591
Fiber optic      No       0.581072
                 Yes      0.418928
No               No       0.925950
                 Yes      0.074050
Name: proportion, dtype: float64


Os segmentos mostram diferenças fortes de comportamento. Clientes com contrato **Month-to-month** têm churn de cerca de **42,71%**, enquanto contratos de **Two year** caem para apenas **2,83%**. Isso sugere que contratos mais longos estão associados a maior retenção.

A permanência também é decisiva: clientes com **0 a 12 meses** têm churn próximo de **47,44%**, contra **9,51%** na faixa de **49+ meses**. Ou seja, os primeiros meses parecem ser a janela mais crítica para ações de retenção.

Em `InternetService`, clientes com **Fiber optic** apresentam churn de aproximadamente **41,89%**, bem acima de DSL (**18,96%**) e de clientes sem internet (**7,41%**). Esse recorte merece destaque no dashboard, porque pode indicar maior sensibilidade a preço, qualidade percebida ou perfil de uso nesse grupo.


## 6. Relação entre cobrança e churn

Aqui vamos transformar `Churn` em uma variável numérica (`Churn_num`) para calcular correlações com `MonthlyCharges` e `TotalCharges`. A correlação ajuda a identificar relações lineares simples entre valores de cobrança e cancelamento.


In [15]:
df["Churn_num"] = df["Churn"].map({"No":0, "Yes": 1})
df[['MonthlyCharges', 'TotalCharges', 'Churn_num']].corr()

,MonthlyCharges,TotalCharges,Churn_num
MonthlyCharges,1.000000,0.651174,0.193356
TotalCharges,0.651174,1.000000,-0.198324
Churn_num,0.193356,-0.198324,1.000000


`MonthlyCharges` tem correlação positiva com churn, em torno de **0,193**. Isso sugere que clientes com mensalidades mais altas tendem um pouco mais ao cancelamento, embora a relação não seja forte isoladamente.

Já `TotalCharges` tem correlação negativa com churn, cerca de **-0,198**. Esse sinal faz sentido porque clientes com maior cobrança acumulada geralmente têm mais tempo de casa, e a análise por `tenure_group` mostrou que clientes antigos cancelam menos.

Essas correlações são úteis como pista, mas não explicam causalidade sozinhas. Elas devem ser interpretadas junto com contrato, permanência, serviços contratados e demais variáveis categóricas.


## 7. Exportação da base enriquecida

Por fim, vamos salvar uma versão tratada da base em `../data/df_eda.csv`. Esse arquivo preserva os dados originais já com `TotalCharges` corrigido e com as colunas criadas durante a análise, como `tenure_group` e `Churn_num`.


In [16]:
df.to_csv("../data/df_eda.csv", index=False)